In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
len(text)

1115394

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab size:", vocab_size)
print("Characters:", ''.join(chars))

Vocab size: 65
Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
import torch

In [6]:
data = torch.tensor(encode(text), dtype= torch.long)

data.shape, data.type 


(torch.Size([1115394]), <function Tensor.type>)

In [8]:
text[:100]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'

In [9]:
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [10]:
n = int(0.9*len(data))

train_data = data[:n]
val_data = data[n:]

also called context window, is the number of previous tokens that the model can consider when predicting the next token. A larger block size allows the model to capture longer dependencies in the text, but it also increases the computational complexity and memory requirements. In this case, a block size of 8 means that the model will look at the previous 8 tokens to predict the next one.

In [15]:
block_size = 8 
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [18]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
 
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")
x,y

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


(tensor([18, 47, 56, 57, 58,  1, 15, 47]),
 tensor([47, 56, 57, 58,  1, 15, 47, 58]))

In [23]:
torch.manual_seed(2608)
batch_size = 4
block_size = 8 

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data [i+1:i+block_size+1] for i in ix])
    return x,y

xb, yb = get_batch('train')

xb, yb

(tensor([[ 1, 57, 39, 63,  6,  1, 58, 46],
         [ 1, 47, 52, 42, 43, 43, 42,  6],
         [57, 61, 53, 56, 52,  1, 40, 63],
         [39, 60, 63,  1, 50, 43, 39, 60]]),
 tensor([[57, 39, 63,  6,  1, 58, 46, 53],
         [47, 52, 42, 43, 43, 42,  6,  1],
         [61, 53, 56, 52,  1, 40, 63,  6],
         [60, 63,  1, 50, 43, 39, 60, 43]]))

batch size is the number of sequences that will be processed in parallel during training. A larger batch size can lead to more stable gradient estimates and faster training, but it also requires more memory. In this case, a batch size of 4 means that 4 sequences will be processed simultaneously.
Random batch generation is a technique used to create batches of data for training machine learning models. Instead of using the entire dataset at once, which can be computationally expensive and may not fit into memory, random batch generation selects a subset of the data randomly for each training iteration. This helps in improving the model's generalization by exposing it to different combinations of data points during training.

In [25]:
# for b in range(batch_size):
#     for t in range(block_size):
#         context = xb[b, :t+1]
#         target = yb[b, t]
#         print(f"when input is {context.tolist()} the target: {target}")

In [26]:
import torch.nn as nn 
from torch.nn import functional as F 


BigramLanguageModel is a simple neural network model that predicts the next token in a sequence based on the previous token. It uses an embedding layer to convert tokens into dense vectors, followed by a linear layer that outputs logits for each possible next token. The model is trained using cross-entropy loss, which measures the difference between the predicted probabilities and the actual target tokens.

In [28]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size): 
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx) # (B,T,C) 
        #B = batch size, T = time step, C = channel size (vocab size)

        return logits 

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)
out.shape

torch.Size([4, 8, 65])